# diag2 — 순방향은 위치를 잃고, `b_k` 가 되살리는가

diag0 · diag1 이 확정한 것:

- 역방향 브랜치의 쿼리 위치 출력은 **관측 무관 상수** `b_k` (K=50/100/150 전부 `max|Δ|=0`)
- `out_k = LayerNorm(0.5·(fwd_k + b_k))` 가 **비트 단위 항등식**
- `b_k` 는 위치끼리 거의 **직교** (코사인 ≈ 0), 크기는 순방향의 **0.84배**

**아직 가설인 것 — 이 노트북이 재는 것:**

> 순방향 스택의 쿼리 위치 출력 `fwd_k` 는 K 가 커질수록 서로 구분이 안 되게 되고,
> `b_k` 가 head 직전에 위치를 다시 분리해 준다.

왜 그럴 거라고 보는가 — 디코더 입력은 `[C ; Q]` 이고 쿼리는 `zeros + pos_embed` 다.
스캔이 쿼리 위치에 도달할 즈음 상태는 이미 관측 토큰 M 개를 흡수한 뒤라,
쿼리 하나하나는 **큰 상태 위의 작은 섭동**이다. 그래서 `fwd_k ≈ fwd_{k+1}` 이 되고,
action head 는 "몇 번째 액션인지" 를 구분하지 못한 채 서로 다른 값을 내놔야 한다.

## 네 가지 표현을 같은 자로 잰다

| | |
|---|---|
| `fwd` | 순방향 스택 출력 (쿼리 위치) |
| `b_k` | 역방향 스택이 주는 상수표 |
| `fused` | `LayerNorm(0.5·(fwd + b))` — **head 가 실제로 보는 것** |
| `ablated` | `LayerNorm(0.5·(fwd + 0))` — `b_k` 만 끈 것 |

**`fused` vs `ablated` 가 핵심 대조다.** 체크포인트도 순방향도 그대로고 `b_k` 만
토글하므로 차이는 전부 `b_k` 탓이다.

## 자

- **`cos_adj`** 이웃 위치 간 코사인. 1 에 가까울수록 "구분 안 됨"
- **`eff_rank`** 특이값의 participation ratio. K 개 위치가 실질적으로 몇 개의 서로 다른
  방향을 쓰는가. K 면 완전 분리, 1 이면 전부 같은 방향
- **`/K`** = `eff_rank / K`, **위치 활용률**. K 끼리 비교하려면 이걸 본다

자 자체는 합성 데이터로 검증했다 — 직교 100개 → `100.00`, 전부 같은 벡터 → `1.00`,
공통성분에 지배된 경우 → `1.10`.

## 읽는 법

| 결과 | 뜻 |
|---|---|
| `fwd` 활용률 낮고 K 커질수록 더 낮아짐 | 가설 확인. 긴 스캔이 위치를 지운다 |
| `fused` 활용률 > `ablated` | `b_k` 가 위치를 복원한다 — **8.2점의 메커니즘** |
| 둘이 비슷하면 | 가설 기각. 원인을 다른 데서 찾아야 한다 |

---

> 커널 얘기는 diag0 · diag1 과 같다. `lerobot` 을 커널로 import 하지 않고
> `mamba_ssm` 이 깔린 venv 로 subprocess 호출한 뒤 결과 json 만 읽는다.

## 0) 부팅

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

_h = Path.cwd()
REPO = next(c for c in (_h, *_h.parents)
            if (c / 'notebooks' / 'libero' / 'diag2_position_collapse.py').exists())
sys.path.insert(0, str(REPO / 'notebooks'))

try:
    import common_v23 as v23
    PYTHON = v23.PYTHON
except Exception as e:
    print('common_v23 import 실패, 기본값 사용:', e)
    PYTHON = os.environ.get('LEROBOT_PYTHON') or str(
        Path.home() / 'lerobot_project' / 'lerobot_env' / 'bin' / 'python')

SCRIPT = REPO / 'notebooks' / 'libero' / 'diag2_position_collapse.py'
SHARE = Path(os.environ.get('LEROBOT_OUTPUT',
                            Path.home() / 'lerobot_project' / 'outputs')) / 'final' / 'share' / 'diag2'
SHARE.mkdir(parents=True, exist_ok=True)

print('repo   :', REPO)
print('python :', PYTHON, '  (있음)' if Path(PYTHON).exists() else '  <- 없다!')
print('script :', SCRIPT, '  (있음)' if SCRIPT.exists() else '  <- 없다!')
print('share  :', SHARE)

## 1) 설정

In [ ]:
SEED   = 0
STEP   = 150_000      # -1 이면 최신 체크포인트
TASK   = 'libero_10'
BATCH  = 4
GPU    = '0'
STAMP  = time.strftime('%Y%m%d_%H%M')

ENV = dict(os.environ,
           PYTHONPATH=str(REPO / 'src'),
           HF_HUB_DISABLE_XET='1',
           MPLBACKEND='Agg',
           CUDA_VISIBLE_DEVICES=GPU)

def run_diag2(tags, json_path, acm2=False, errors=False):
    cmd = [PYTHON, str(SCRIPT), '--tags', tags, '--seed', str(SEED), '--step', str(STEP),
           '--task', TASK, '--batch', str(BATCH),
           '--save-dir', str(SHARE), '--json', str(json_path)]
    if acm2:
        cmd.append('--acm2')
    if errors:
        cmd.append('--errors')
    print('$', ' '.join(cmd), '\n')
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=ENV, cwd=str(REPO))
    for line in p.stdout:
        print(line, end='')
    print(f'\n[exit {p.wait()}]')
    return json.loads(Path(json_path).read_text(encoding='utf-8')) if Path(json_path).exists() else {}

## 2) K=100 단일 — 이 셀이 가설의 답이다

`fused` 의 위치 활용률이 `ablated` 보다 확실히 높으면 확인된 것이다.

In [ ]:
res100 = run_diag2('bimamba_pure', SHARE / f'diag2_k100_{STAMP}.json', errors=True)

## 3) K 50 / 100 / 150 + 단방향 기준선

ACM2 체크포인트도 같이 잰다. ACM2 는 역방향이 없으니 `fwd` 하나뿐이고,
**56.8 을 낸 모델의 위치 활용률이 얼마인지**가 비교 기준이 된다.

In [ ]:
results = run_diag2('all', SHARE / f'diag2_all_{STAMP}.json', acm2=True, errors=True)
print('\n받은 태그:', list(results))

## 4) 요약표

In [ ]:
import csv

K_OF = {'bimamba_pure_k50': 50, 'bimamba_pure': 100, 'bimamba_pure_k150': 150,
        'acm2_k50': 50, 'acm2': 100, 'acm2_k150': 150}
order = [(t, r) for t, r in results.items() if r.get('ok')]
order.sort(key=lambda kv: (not kv[1].get('bimamba'), K_OF.get(kv[0], 0)))

hdr = f"{'K':>4} {'tag':<20} {'표현':<9} {'이웃cos':>9} {'eff_rank':>10} {'활용률':>8}"
print(hdr); print('-' * len(hdr))
rows = []
for tag, r in order:
    for name in ('fwd', 'b', 'ablated', 'fused'):
        s = r['sep'].get(name)
        if not s:
            continue
        print(f"{r['K']:>4} {tag:<20} {name:<9} {s['cos_adj_mean']:>9.3f} "
              f"{s['eff_rank']:>10.1f} {s['use_ratio']:>8.2f}")
        rows.append({'K': r['K'], 'tag': tag, 'rep': name,
                     'cos_adj_mean': s['cos_adj_mean'],
                     'cos_offdiag_mean': s['cos_offdiag_mean'],
                     'eff_rank': s['eff_rank'],
                     'eff_rank_centered': s['eff_rank_centered'],
                     'use_ratio': s['use_ratio']})
    print('-' * len(hdr))

if rows:
    csv_path = SHARE / f'diag2_summary_{STAMP}.csv'
    with csv_path.open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
    print('saved', csv_path)

## 5) 핵심 그림 — 위치 활용률이 K 에 따라 어떻게 되나

**이 그림 하나가 논문에 들어갈 재료다.**

- `fwd` 가 아래로 처지고 K 커질수록 더 처지면 -> 긴 스캔이 위치를 지운다
- `fused` 가 `ablated` 위로 올라오면 -> `b_k` 가 복원한다

In [ ]:
import matplotlib.pyplot as plt

bim = [(t, r) for t, r in order if r.get('bimamba')]
acm = [(t, r) for t, r in order if not r.get('bimamba')]

styles = {'fwd':     ('o-',  'tab:blue',   'forward branch'),
          'ablated': ('s--', 'tab:orange', 'output, b_k off'),
          'fused':   ('^-',  'tab:red',    'output, b_k on'),
          'b':       ('d:',  'tab:green',  'b_k table')}

fig, ax = plt.subplots(figsize=(7, 4.4))
for name, (mk, c, lab) in styles.items():
    xs = [r['K'] for _, r in bim if name in r['sep']]
    ys = [r['sep'][name]['use_ratio'] for _, r in bim if name in r['sep']]
    if xs:
        ax.plot(xs, ys, mk, color=c, label=lab, lw=1.8, ms=7)
if acm:
    ax.plot([r['K'] for _, r in acm], [r['sep']['fwd']['use_ratio'] for _, r in acm],
            'v-.', color='grey', label='ACM2 (unidirectional)', lw=1.6, ms=7)
ax.set_xlabel('chunk length K'); ax.set_ylabel('position utilisation  (eff_rank / K)')
ax.set_title('Do query positions stay distinguishable?')
ax.set_ylim(0, 1.05); ax.grid(alpha=.3); ax.legend(fontsize=9)
fig.tight_layout()
png = SHARE / f'diag2_use_ratio_{STAMP}.png'
fig.savefig(png, dpi=150, bbox_inches='tight')
print('saved', png)
plt.show()

## 6) 위치별 이웃 코사인 — 청크 안 어디서 뭉개지나

In [ ]:
n = len(bim)
if n:
    fig, axes = plt.subplots(1, n, figsize=(4.3 * n, 3.8), squeeze=False, sharey=True)
    for j, (tag, r) in enumerate(bim):
        ax = axes[0][j]
        for name, (mk, c, lab) in styles.items():
            s = r['sep'].get(name)
            if not s:
                continue
            y = s['cos_adj']
            ax.plot(range(1, len(y) + 1), y, color=c, label=lab, lw=1.3)
        ax.set_title(f"K={r['K']}"); ax.set_xlabel('position k')
        ax.axhline(0, color='k', lw=.5); ax.grid(alpha=.3); ax.set_ylim(-0.4, 1.05)
        if j == 0:
            ax.set_ylabel('cos(x_k, x_k+1)'); ax.legend(fontsize=8)
    fig.tight_layout()
    png2 = SHARE / f'diag2_cos_adj_{STAMP}.png'
    fig.savefig(png2, dpi=150, bbox_inches='tight')
    print('saved', png2)
    plt.show()

## 7) 위치별 예측 오차 — 9/14 에 원래 하려던 것

같은 체크포인트에서 `b_k` 만 껐다 켰다 하며 청크 안 위치별 오차를 잰다.
`b_k` 를 끄면 오차가 오르고, **청크 뒤로 갈수록 더 많이 오르면** 위치 붕괴 가설과 맞는다.

In [ ]:
have_err = [(t, r) for t, r in bim if r.get('errors', {}).get('ok')]
if not have_err:
    print('오차 결과가 없다. errors=True 로 다시 돌릴 것.')
else:
    fig, axes = plt.subplots(1, len(have_err), figsize=(4.3 * len(have_err), 3.8), squeeze=False)
    for j, (tag, r) in enumerate(have_err):
        e = r['errors']; ax = axes[0][j]
        K = len(e['err_on'])
        ax.plot(range(1, K + 1), e['err_off'], color='tab:orange', lw=1.3, label='b_k off')
        ax.plot(range(1, K + 1), e['err_on'], color='tab:red', lw=1.3, label='b_k on')
        ax.set_title(f"K={r['K']}"); ax.set_xlabel('position k'); ax.grid(alpha=.3)
        if j == 0:
            ax.set_ylabel('mean |pred - target|'); ax.legend(fontsize=9)
    fig.tight_layout()
    png3 = SHARE / f'diag2_pos_error_{STAMP}.png'
    fig.savefig(png3, dpi=150, bbox_inches='tight')
    print('saved', png3)
    plt.show()

## 8) 결과 읽는 법

### 가설 확인 (셀 5)

**`fwd` 활용률이 낮고 K 가 커질수록 더 낮아짐 + `fused` > `ablated`** -> 확인.
"긴 스캔이 위치 정체성을 지우고, 역방향 브랜치가 head 직전에 상수표로 되살린다" 가
8.2 점의 메커니즘이다. 이게 나오면 논문 §5 에 analysis 소절로 바로 들어간다.

**`fused` ≈ `ablated`** -> 기각. `b_k` 가 위치 분리에 기여하지 않는다는 뜻이므로
8.2 점의 원인을 다른 데서 찾아야 한다. 이 경우 `bimamba_scan='random'` 을
돌려서 "역방향 순서" 자체에 뭐가 있는지 보는 게 다음 수다.

**`fwd` 활용률이 원래부터 높음** -> 위치가 안 지워진다는 뜻. 가설이 틀렸고
`b_k` 는 위치 복원이 아니라 다른 일을 한다.

### ACM2 대조 (셀 4·5)

ACM2 는 `fwd` 하나뿐이고 56.8 을 냈다. BiMamba 의 `ablated` 와 활용률이 비슷하게
나와야 앞뒤가 맞는다 — 둘 다 "b_k 없는 순방향" 이기 때문이다.
크게 다르면 두 체크포인트의 순방향 스택이 서로 다르게 학습됐다는 뜻이라
`b_k` 만의 효과로 해석하면 안 된다.

### 위치별 오차 (셀 7)

`b_k` 를 끄면 오차가 오르는데 **뒤쪽 위치에서 더 많이 오르면** 그림이 완성된다.
평평하게 오르면 `b_k` 는 위치별이 아니라 전역적으로 돕는다는 뜻이다.

⚠️ 정답 크기와 예측 크기가 5배 이상 차이나면 정규화 공간이 어긋난 것이다.
스크립트가 경고를 찍으니 그때는 오차 숫자를 쓰면 안 된다.

### 한계

- seed 0 / step 150k / libero_10 / batch 4 단일 세팅
- `use_action_self_attention=True` 면 `fused`·`ablated` 가 근사다 (스크립트가 경고)
- 활용률은 표현의 성질이지 성능이 아니다. 성능과의 연결은 56.8 / 56.0 / 65.0 삼각비교가 한다

산출물은 `outputs/final/share/diag2/` 에 json · csv · npz · png 로 남는다.